# CropGuard AI: YOLO11s Agricultural Pest Detection Model Training Workflow

**Project:** CropGuard AI — Intelligent Pest Detection & Crop Protection  
**Model Architecture:** YOLO11s (Ultralytics)  
**Target Classes (6):** `Aphids`, `Whitefly`, `Brown Planthopper`, `Yellow Stem Borer`, `Thrips`, `Helicoverpa armigera`  
**Execution Environment:** Google Colab GPU (NVIDIA Tesla T4 / A100)  

This notebook downloads the verified bounding-box pest dataset, validates annotations, fine-tunes pretrained YOLO11s, generates evaluation metrics (Precision, Recall, F1, mAP@50, mAP@50:95, Confusion Matrix), and exports `best.pt` and `best.onnx`.

## 1. Verify GPU Acceleration & Install Dependencies

In [ ]:
!nvidia-smi
!pip install --upgrade ultralytics opencv-python-headless matplotlib seaborn pillow pyyaml

In [ ]:
import torch
import os
import yaml
from ultralytics import YOLO

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## 2. Dataset Acquisition & YOLO Format Preparation
The dataset contains verified bounding-box annotations for our 6 target agricultural pests across Paddy/Rice, Cotton, Tomato, and Chilli crops.

In [ ]:
import os
import zipfile
import urllib.request

dataset_dir = "/content/cropguard_dataset"
os.makedirs(dataset_dir, exist_ok=True)

# Dataset configuration dictionary
dataset_config = {
    'path': dataset_dir,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 6,
    'names': {
        0: 'Aphids',
        1: 'Whitefly',
        2: 'Brown Planthopper',
        3: 'Yellow Stem Borer',
        4: 'Thrips',
        5: 'Helicoverpa armigera'
    }
}

yaml_path = "/content/cropguard_data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"Dataset configuration written to {yaml_path}:")
with open(yaml_path) as f:
    print(f.read())

## 3. Fine-Tune YOLO11s Model with Transfer Learning
Fine-tune `yolo11s.pt` with early stopping (patience=15), image resolution 640x640, and field augmentations.

In [ ]:
# Load pretrained YOLO11s model
model = YOLO('yolo11s.pt')

# Start training
results = model.train(
    data=yaml_path,
    epochs=75,
    batch=16,
    imgsz=640,
    patience=15,
    device=0,
    optimizer='auto',
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    project='cropguard_yolo',
    name='yolo11s_run',
    exist_ok=True,
    verbose=True
)

## 4. Model Evaluation & Performance Metrics
Evaluate on held-out test data for Precision, Recall, F1, mAP@50, and mAP@50:95.

In [ ]:
# Validate the best trained model checkpoint
best_weights = '/content/cropguard_yolo/yolo11s_run/weights/best.pt'
trained_model = YOLO(best_weights)

metrics = trained_model.val(data=yaml_path, split='test', device=0)

print("=" * 50)
print(f"mAP @ 0.50     : {metrics.box.map50 * 100:.2f}%")
print(f"mAP @ 0.50:0.95: {metrics.box.map * 100:.2f}%")
print(f"Mean Precision : {metrics.box.mp * 100:.2f}%")
print(f"Mean Recall    : {metrics.box.mr * 100:.2f}%")
print("=" * 50)

## 5. Display Evaluation Curves & Confusion Matrix

In [ ]:
from IPython.display import Image, display
import glob

eval_plots = [
    '/content/cropguard_yolo/yolo11s_run/confusion_matrix.png',
    '/content/cropguard_yolo/yolo11s_run/results.png',
    '/content/cropguard_yolo/yolo11s_run/PR_curve.png',
    '/content/cropguard_yolo/yolo11s_run/F1_curve.png'
]

for plot_path in eval_plots:
    if os.path.exists(plot_path):
        print(f"\nDisplaying: {os.path.basename(plot_path)}")
        display(Image(plot_path))

## 6. Export Optimized Weights & Download to Local Machine

In [ ]:
# Export to ONNX runtime format
trained_model.export(format='onnx', dynamic=False, simplify=True)

# Download to local laptop for backend integration
from google.colab import files
files.download(best_weights)
onnx_path = best_weights.replace('.pt', '.onnx')
if os.path.exists(onnx_path):
    files.download(onnx_path)